# NOAA Weather Data: Exploration, Cleaning, and Setup

This notebook covers the first findings, setup, and cleaning of the NOAA weather data that we will be using alongside our car crash data for the Prince George's County area.

Our goal is to end up with the amount of precipitation per day for Prince George's County overall, for the dates 2019-2023.

We will be using pandas (imported as pd) to clean the data, along with numpy (imported as np). We are loading the NOAA weather data for the Prince George's County region into a pandas DataFrame.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the raw NOAA CSV
weather_raw = pd.read_csv("Precipitation2015-2020.csv")

## First Look at the Raw Data

We will now take a first look at the raw data, before changing anything. This is often called initial data inspection or exploratory data analysis. We will use pandas functions such as .head(), .info(), .describe(), and .isna().sum() to check the size of the data, the column types, the missing values, and the range of the values, so we can spot anything that looks wrong or messy.

In [3]:
weather_raw.head()

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,DAPR,MDPR,PRCP,SNOW,SNWD
0,US1MDPG0073,"UPPER MARLBORO 1.7 N, MD US",38.842453,-76.752502,44.2,2015-04-06,NaN,NaN,0.00,0.0,NaN
1,US1MDPG0073,"UPPER MARLBORO 1.7 N, MD US",38.842453,-76.752502,44.2,2015-04-07,NaN,NaN,0.03,NaN,NaN
2,US1MDPG0073,"UPPER MARLBORO 1.7 N, MD US",38.842453,-76.752502,44.2,2015-04-08,NaN,NaN,0.06,NaN,NaN
3,US1MDPG0073,"UPPER MARLBORO 1.7 N, MD US",38.842453,-76.752502,44.2,2015-04-09,NaN,NaN,0.05,NaN,NaN
4,US1MDPG0073,"UPPER MARLBORO 1.7 N, MD US",38.842453,-76.752502,44.2,2015-04-10,NaN,NaN,0.05,NaN,NaN


In [4]:
weather_raw.shape

(51046, 11)

In [5]:
weather_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 51046 entries, 0 to 51045
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   STATION    51046 non-null  str    
 1   NAME       51046 non-null  str    
 2   LATITUDE   51046 non-null  float64
 3   LONGITUDE  51046 non-null  float64
 4   ELEVATION  51046 non-null  float64
 5   DATE       51046 non-null  str    
 6   DAPR       814 non-null    float64
 7   MDPR       811 non-null    float64
 8   PRCP       49532 non-null  float64
 9   SNOW       29398 non-null  float64
 10  SNWD       8831 non-null   float64
dtypes: float64(8), str(3)
memory usage: 4.3 MB


In [6]:
weather_raw.describe()

,LATITUDE,LONGITUDE,ELEVATION,DAPR,MDPR,PRCP,SNOW,SNWD
count,51046.000000,51046.000000,51046.000000,814.000000,811.000000,49532.000000,29398.000000,8831.000000
mean,38.923028,-76.873867,57.965555,5.477887,0.695290,0.137899,0.055225,0.423791
std,0.136056,0.098596,45.480931,5.473306,0.922346,0.364326,0.506977,1.624508
min,38.665455,-77.028948,7.300000,2.000000,0.000000,0.000000,0.000000,0.000000
25%,38.822430,-76.931940,30.800000,2.000000,0.050000,0.000000,0.000000,0.000000
50%,38.968891,-76.902013,44.500000,3.000000,0.370000,0.000000,0.000000,0.000000
75%,39.030200,-76.768300,65.200000,7.000000,1.000000,0.070000,0.000000,0.000000
max,39.117498,-76.678062,248.100000,62.000000,8.680000,8.360000,17.400000,24.000000


In [7]:
weather_raw.isna().sum()

STATION          0
NAME             0
LATITUDE         0
LONGITUDE        0
ELEVATION        0
DATE             0
DAPR         50232
MDPR         50235
PRCP          1514
SNOW         21648
SNWD         42215
dtype: int64

In [8]:
weather_raw["DATE"].agg(["min", "max"])

min    2014-01-01
max    2026-01-01
Name: DATE, dtype: str

Things we need to fix, in order:

1. I noticed the dates are stored as text and not as a datetime type. We also have to filter by date to 2019-2023 to get our desired range.

In [9]:
# Work on a copy so weather_raw stays untouched
weather = weather_raw.copy()

# Convert DATE from text to datetime
weather["DATE"] = pd.to_datetime(weather["DATE"])

In [10]:
weather["DATE"].dtype

dtype('<M8[us]')

In [11]:
# Keep only dates from 2019-01-01 to 2023-12-31, then renumber the rows
weather = weather[(weather["DATE"] >= "2019-01-01") & (weather["DATE"] <= "2023-12-31")].reset_index(drop=True)

In [12]:
# Confirm row/column counts and the new date range
print(weather.shape)
weather["DATE"].agg(["min", "max"])

(20106, 11)


min   2019-01-01
max   2023-12-31
Name: DATE, dtype: datetime64[us]

2. Before renaming, we will drop the columns we do not need, so we do not spend time cleaning and renaming them.

Columns we are dropping and why:

- **SNOW and SNWD (snowfall and snow depth):** the precipitation column already includes snow, measured as melted water. Snow only shows up on about 2% of the days (43 of 1,826), and the columns are mostly blank (44% and 85%).
- **DAPR and MDPR (multi-day totals):** these are rain totals spread over several days, so they cannot be assigned to a single date and we cannot use them for the daily county average. The rows they sit on have a blank precipitation value, which we will deal with when we handle the missing precipitation values.

In [13]:
# Drop snow columns and multi-day total columns
weather = weather.drop(columns=["SNOW", "SNWD", "DAPR", "MDPR"])

In [14]:
weather.columns

Index(['STATION', 'NAME', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'DATE',
       'PRCP'],
      dtype='str')

3. The next step is to rename the columns, because the naming is a little unclear.

In [15]:
# Rename columns to clear names with units
weather = weather.rename(columns={
    "STATION": "station_id",
    "NAME": "station_name",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "ELEVATION": "elevation_meters",
    "DATE": "date",
    "PRCP": "precipitation_inches",
})

In [16]:
weather.head()

,station_id,station_name,latitude,longitude,elevation_meters,date,precipitation_inches
0,US1MDPG0050,"BOWIE 3.9 S, MD US",38.903576,-76.725861,38.7,2019-01-03,NaN
1,US1MDPG0050,"BOWIE 3.9 S, MD US",38.903576,-76.725861,38.7,2019-01-05,0.14
2,US1MDPG0050,"BOWIE 3.9 S, MD US",38.903576,-76.725861,38.7,2019-01-06,0.02
3,US1MDPG0050,"BOWIE 3.9 S, MD US",38.903576,-76.725861,38.7,2019-01-08,0.11
4,US1MDPG0050,"BOWIE 3.9 S, MD US",38.903576,-76.725861,38.7,2019-01-09,0.17


4. Next we check the blank precipitation values, to see how many there are and whether it is safe to drop them.

In [17]:
# Count blank precipitation values and their share of all rows
blank_count = weather["precipitation_inches"].isna().sum()
total_rows = len(weather)
print(f"Blank precipitation rows: {blank_count} of {total_rows} ({blank_count / total_rows:.1%})")

Blank precipitation rows: 214 of 20106 (1.1%)


Only 214 of the 20,106 rows (1.1%) have a blank precipitation value. That is negligible compared to the rest of the data, so it is ok to drop them.

**Why they are possibly missing:** the file does not say, but the pattern suggests two causes.

- Most of them (175) were rows where the rain was reported as a multi-day total instead of a daily value, probably because the gauge was not read for several days.
- Most of the rest (39) come from one station, Greenbelt 1.0 N, which appears to report snow but not rain.

In [18]:
# Drop rows with a blank precipitation value
weather = weather.dropna(subset=["precipitation_inches"]).reset_index(drop=True)

In [19]:
# Confirm rows left and that no blanks remain
print(weather.shape)
weather["precipitation_inches"].isna().sum()

(19892, 7)


np.int64(0)

5. We also know some stations are missing days entirely: there are about 7,000 station-days where a station was running but has no row for that day. We are not doing anything about these in the cleaning, because the missing days are not in the data to fix, and filling them in would invent rain (a 0 would wrongly say it was a dry day). Every day in 2019-2023 still has readings from at least 5 stations. We will account for this later, when we combine the stations, by keeping track of how many stations reported each day.

Column guide for weather (cleaned station data). Each row is one station on one day.

- **station_id**: NOAA's ID for the station (one rain gauge at one fixed location).
- **station_name**: the station's name, which is the nearest town plus a distance and direction from it. It is only a label, not the area the station covers.
- **latitude**: latitude of the station.
- **longitude**: longitude of the station.
- **elevation_meters**: height of the station above sea level, in meters.
- **date**: the day of the reading, 2019-01-01 to 2023-12-31.
- **precipitation_inches**: daily precipitation in inches. Snow is included, measured as melted water.

6. Now we create the county average table. It has one row per day, with the average precipitation across all the stations that reported that day, and how many stations reported.

In [20]:
# One row per day: mean precipitation across stations, and how many stations reported
county_daily = (
    weather.groupby("date")["precipitation_inches"]
    .agg(precipitation_mean_inches="mean", n_stations="count")
    .reset_index()
)

In [21]:
print(county_daily.shape)
county_daily.head()

(1826, 3)


,date,precipitation_mean_inches,n_stations
0,2019-01-01,0.514167,12
1,2019-01-02,0.001538,13
2,2019-01-03,0.004167,12
3,2019-01-04,0.000000,11
4,2019-01-05,0.070000,12


7. We also add two more columns to the county table: the heaviest single-station reading each day, and the share of stations that recorded any rain.

In [22]:
# Add the daily maximum and the share of stations that recorded rain (> 0 inches)
extra = weather.groupby("date")["precipitation_inches"].agg(
    precipitation_max_inches="max",
    share_stations_wet=lambda s: (s > 0).mean(),
).reset_index()
county_daily = county_daily.merge(extra, on="date")

# Put the columns in a clear order
county_daily = county_daily[[
    "date", "precipitation_mean_inches", "precipitation_max_inches", "share_stations_wet", "n_stations"
]]

In [23]:
print(county_daily.shape)
county_daily.head()

(1826, 5)


,date,precipitation_mean_inches,precipitation_max_inches,share_stations_wet,n_stations
0,2019-01-01,0.514167,0.69,1.000000,12
1,2019-01-02,0.001538,0.01,0.153846,13
2,2019-01-03,0.004167,0.02,0.333333,12
3,2019-01-04,0.000000,0.00,0.000000,11
4,2019-01-05,0.070000,0.14,1.000000,12


Column guide for county_daily. There is one row per day for the whole county.

- **date**: the day, from 2019-01-01 to 2023-12-31.
- **precipitation_mean_inches**: average precipitation in inches across all the stations that reported that day.
- **precipitation_max_inches**: the heaviest single-station precipitation reading that day, in inches.
- **share_stations_wet**: the share of reporting stations that recorded any rain (more than 0 inches). 0 means no station had rain, 1 means every station did.
- **n_stations**: how many stations reported that day. Days with fewer stations are less reliable.

In [24]:
# Save the county table to a CSV file (without the row numbers)
county_daily.to_csv("county_daily_precipitation_2019_2023.csv", index=False)